# GPFL Client

> The core abstraction for `GPFL` client: [https://arxiv.org/pdf/2308.10279v3](https://arxiv.org/pdf/2308.10279v3)

In [ ]:
#| default_exp clients.gpfl

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import copy
import random
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from fedai.clients.base_client import BaseClient
from fedai.utils.registery import AlgorithmRegistry

<torch._C.Generator>

In [ ]:
#| export
@AlgorithmRegistry.register_client("gpfl")
class ClientGPFL(BaseClient):
    def __init__(self,
                 id, # Unique identifier for the client
                 cfg, # A configuration object containing all the necessary parameters for training and evaluation.
                 train_loader, 
                 test_loader,
                 state, # A dictionary containing the model, optimizer and any changing component needed.
                 criterion,
                 device= None,
                 t= 0,
                 **kwargs # extra client-specific parameters (that cannot be passed in state and cfg), can be passed as here.
                 ):  
                 
        super().__init__(id, cfg, train_loader, test_loader, state, criterion, device, t, **kwargs)
        
        if not hasattr(self, "sample_per_class"):
            self.sample_per_class = torch.zeros(self.cfg.data.num_classes, dtype=torch.long)
            for batch in self.train_loader:
                for yy in batch[self.label_key]:
                    self.sample_per_class[yy.item()] += 1
            self.sample_per_class = self.sample_per_class / torch.sum(self.sample_per_class)


### Training

In [ ]:
#| export
@patch
def fit(self: ClientGPFL):
    
    self.model.to(self.device)
    self.model.train()
    self.GCE.to(self.device)
    self.GCE.train()
    self.CoV.to(self.device)
    self.CoV.train()
    self.personalized_conditional_input = self.personalized_conditional_input.to(self.device)
    self.generic_conditional_input = self.generic_conditional_input.to(self.device)
   
    for _ in range(self.cfg.local_epochs):
        for i, batch in enumerate(self.train_loader):
            self.optimizer.zero_grad()

            batch = self.send_to_device(batch)
            X, y = batch[self.data_key], batch[self.label_key]
            feat = self.model.backbone(X)

            feat_P = self.CoV(feat, self.personalized_conditional_input)
            output = self.model.head(feat_P)

            feat_G = self.CoV(feat, self.generic_conditional_input)
            softmax_loss = self.GCE(feat_G, y)

            loss = self.criterion(output, y)
            loss += softmax_loss

            # emb = torch.zeros_like(feat)
            # for i, yy in enumerate(y):
            #     emb[i, :] = self.GCE_frozen.embedding(yy).detach().data
            # loss += torch.norm(feat_G - emb, 2) * self.lamda

            self.optimizer.zero_grad()
            self.GCE_optimizer.zero_grad()
            self.CoV_optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            self.GCE_optimizer.step()
            self.CoV_optimizer.step()


### Evaluation

In [ ]:
#| export
@patch
def test_stats(self: ClientGPFL, batch: dict) -> tuple:
    metrics = {k: 0 for k in list(self.cfg.training_metrics)}  # Ensure metrics is always defined

    X, y = batch[self.data_key], batch[self.label_key]
    feat = self.model.backbone(X)
    feat_P = self.CoV(feat, self.personalized_conditional_input)
    output = self.model.head(feat_P)
    loss = self.criterion(output, y)
    y_pred = output.argmax(dim=-1)
    y_true = batch[self.label_key]

    metrics = self.training_metrics.compute(y_pred= y_pred, y_true= y_true)

    return loss, metrics


In [ ]:
#| export
@patch
def train_stats(self: ClientGPFL, batch: dict) -> tuple:
    metrics = {k: 0 for k in list(self.cfg.training_metrics)}  # Ensure metrics is always defined

    X, y = batch[self.data_key], batch[self.label_key]
    feat = self.model.backbone(X)
    feat_P = self.CoV(feat, self.personalized_conditional_input)
    output = self.model.head(feat_P)

    feat_G = self.CoV(feat, self.generic_conditional_input)
    softmax_loss = self.GCE(feat_G, y)

    loss = self.criterion(output, y)
    loss += softmax_loss

    y_pred = output.argmax(dim=-1)
    y_true = batch[self.label_key]

    metrics = self.training_metrics.compute(y_pred= y_pred, y_true= y_true)

    return loss, metrics

In [ ]:
#| export
@patch
def evaluate_local(self: ClientGPFL, loader= 'train') -> dict:
    total_loss = 0
    lst_metrics = []

    self.model = self.model.to(self.device)
    self.model.eval()
    self.GCE = self.GCE.to(self.device)
    self.GCE.eval()
    self.CoV = self.CoV.to(self.device)
    self.CoV.eval()
    self.personalized_conditional_input = self.personalized_conditional_input.to(self.device)
    self.generic_conditional_input = self.generic_conditional_input.to(self.device)
    
    num_eval = 0
    data_loader = self.train_loader if loader == 'train' else self.test_loader
    
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            batch = self.send_to_device(batch)
            loss, metrics = self.test_stats(batch) if loader == 'test' else self.train_stats(batch)          
            if not math.isnan(loss.item()):
                total_loss += loss.item()  
                num_eval += len(batch[self.data_key])  # Ensure num_eval is updated
                lst_metrics.append(metrics)           
    
    avg_loss = total_loss / num_eval if num_eval > 0 else 0.0
    self.logger.info(f"Average {loader} Loss is : {avg_loss}")
    
    if lst_metrics:
        total_metrics = {k: sum(m.get(k, 0) for m in lst_metrics) / len(lst_metrics) for k in self.cfg.test_metrics}
    else:
        total_metrics = {k: 0.0 for k in self.cfg.test_metrics}

    self.logger.info(f"Average {loader} Metrics: {total_metrics}")
    return {"loss": avg_loss, "metrics": total_metrics}


### Saving State

In [ ]:
#| export
@patch
def save_state(self: ClientGPFL, save_to_disk= False):  

    state_dict = {}
    state_dict['model'] = self.model.state_dict()
    state_dict['GCE'] = self.GCE.state_dict()
    state_dict['CoV'] = self.CoV.state_dict()
    state_dict['GCE_frozen'] = self.GCE_frozen.state_dict()

    state_dict['optimizer'] = self.optimizer.state_dict()
    state_dict['GCE_optimizer'] = self.GCE_optimizer.state_dict()
    state_dict['CoV_optimizer'] = self.CoV_optimizer.state_dict()

    state_dict['personalized_conditional_input'] = self.personalized_conditional_input
    state_dict['generic_conditional_input'] = self.generic_conditional_input
    state_dict['sample_per_class'] = self.sample_per_class

    if save_to_disk:
        state_path = os.path.join(self.cfg.server.save_dir, str(self.t), f"local_output_{self.id}", "state.pth")
        if not os.path.exists(os.path.dirname(state_path)):
            os.makedirs(os.path.dirname(state_path))

        torch.save(state_dict, state_path)

    return state_dict

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()